In [2]:
from pathlib import Path
import pickle

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


MODELS_DIR = PROJECT_ROOT / "models"
EXPERIMENT_MODELS_DIR = MODELS_DIR / "experiments" / "no_conf"

EXPERIMENT_MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Experiment model output:", EXPERIMENT_MODELS_DIR)


Project root: /home/user1/project/fifa_prediction
Experiment model output: /home/user1/project/fifa_prediction/models/experiments/no_conf


In [3]:
df = pd.read_csv( "/home/user1/project/fifa_prediction/processed/df_match_features.csv")

print(df.shape)
display(df.head())
display(df.columns.tolist())
display(df.tail())

(49122, 16)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000,0.000000,1,False,UEFA,UEFA
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,1502.801300,1497.198700,5.602600,1,False,UEFA,UEFA
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,1486.622531,1513.377469,-26.754938,1,False,UEFA,UEFA
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,1505.454946,1494.545054,10.909891,1,False,UEFA,UEFA
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,1497.633108,1502.366892,-4.733783,1,False,UEFA,UEFA


['date',
 'home_team',
 'away_team',
 'home_score',
 'away_score',
 'tournament',
 'city',
 'country',
 'neutral',
 'home_elo_pre',
 'away_elo_pre',
 'elo_diff',
 'tournament_weight',
 'is_competitive',
 'home_confederation',
 'away_confederation']

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
49117,2026-06-01,Norway,Sweden,3.0,1.0,Friendly,Oslo,Norway,False,1961.224460,1771.742825,189.481635,1,False,UEFA,UEFA
49118,2026-06-01,Turkey,North Macedonia,4.0,0.0,Friendly,Istanbul,Turkey,False,1954.227181,1646.591037,307.636144,1,False,UEFA,Unknown
49119,2026-06-01,Slovakia,Malta,2.0,1.0,Friendly,Dunajská Streda,Slovakia,False,1725.366386,1326.515389,398.850996,1,False,Unknown,Unknown
49120,2026-06-01,Maldives,Afghanistan,0.0,1.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False,1036.288233,1225.594413,-189.306181,1,True,Unknown,Unknown
49121,2026-06-01,Maldives,Pakistan,0.0,3.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False,1036.288233,1030.559852,5.728380,1,True,Unknown,Unknown


In [4]:
required_columns = [
    "home_score",
    "away_score",
    "home_elo_pre",
    "away_elo_pre",
    "elo_diff",
    "tournament_weight",
    "neutral",
]

missing_columns = [col for col in required_columns if col not in df.columns]

print("Missing columns:", missing_columns)


Missing columns: []


In [6]:
train_df = df[df['is_competitive']].copy()
print(f"Training on {len(train_df):,} competitive matches")


Training on 30,837 competitive matches


In [7]:
train_df = df[df['is_competitive'] & (df['date'] >= '2000-01-01')].copy()


In [8]:
display(train_df.head())
display(train_df.tail())

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
24082,2000-01-22,Ghana,Cameroon,1.0,1.0,African Cup of Nations,Accra,Ghana,False,1638.456967,1701.198916,-62.741949,4,True,CAF,Unknown
24083,2000-01-23,China,Philippines,8.0,0.0,AFC Asian Cup qualification,Hanoi,Vietnam,True,1776.265578,928.707445,847.558133,3,True,Unknown,Unknown
24084,2000-01-23,Egypt,Zambia,2.0,0.0,African Cup of Nations,Kano,Nigeria,True,1683.189245,1671.610112,11.579133,4,True,CAF,Unknown
24086,2000-01-23,Nigeria,Tunisia,4.0,2.0,African Cup of Nations,Lagos,Nigeria,False,1691.371221,1682.762992,8.608229,4,True,Unknown,CAF
24087,2000-01-23,South Africa,Gabon,3.0,1.0,African Cup of Nations,Kumasi,Ghana,True,1687.805918,1586.760996,101.044922,4,True,CAF,Unknown


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,is_competitive,home_confederation,away_confederation
49091,2026-05-27,India,Jamaica,0.0,2.0,Unity Cup,London,England,True,1250.225947,1675.127514,-424.901567,1,True,Unknown,Unknown
49101,2026-05-30,Zimbabwe,India,1.0,0.0,Unity Cup,London,England,True,1456.169271,1248.431872,207.737399,1,True,Unknown,Unknown
49102,2026-05-30,Nigeria,Jamaica,3.0,0.0,Unity Cup,London,England,True,1868.347457,1676.921589,191.425868,1,True,Unknown,Unknown
49120,2026-06-01,Maldives,Afghanistan,0.0,1.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False,1036.288233,1225.594413,-189.306181,1,True,Unknown,Unknown
49121,2026-06-01,Maldives,Pakistan,0.0,3.0,Diamond Jubilee International Football Tournament,Malé,Maldives,False,1036.288233,1030.559852,5.728380,1,True,Unknown,Unknown


In [12]:
feature_cols_no_conf = [
    "home_elo_pre",
    "away_elo_pre",
    "elo_diff",
    "tournament_weight",
    "neutral",
]

target_home = "home_score"
target_away = "away_score"

df_model = train_df.dropna(
    subset=feature_cols_no_conf + [target_home, target_away]
).copy()

X = df_model[feature_cols_no_conf].copy()
X = X.astype(float)

X = sm.add_constant(X, has_constant="add")

y_home = df_model[target_home].astype(float)
y_away = df_model[target_away].astype(float)

print("X shape:", X.shape)
print("Home target shape:", y_home.shape)
print("Away target shape:", y_away.shape)

display(X.head())

X shape: (16715, 6)
Home target shape: (16715,)
Away target shape: (16715,)


,const,home_elo_pre,away_elo_pre,elo_diff,tournament_weight,neutral
24082,1.0,1638.456967,1701.198916,-62.741949,4.0,0.0
24083,1.0,1776.265578,928.707445,847.558133,3.0,1.0
24084,1.0,1683.189245,1671.610112,11.579133,4.0,1.0
24086,1.0,1691.371221,1682.762992,8.608229,4.0,0.0
24087,1.0,1687.805918,1586.760996,101.044922,4.0,1.0


In [13]:
X.columns.tolist()


['const',
 'home_elo_pre',
 'away_elo_pre',
 'elo_diff',
 'tournament_weight',
 'neutral']

In [14]:
model_home_no_conf = sm.GLM(
    y_home,
    X,
    family=sm.families.Poisson()
).fit()

model_away_no_conf = sm.GLM(
    y_away,
    X,
    family=sm.families.Poisson()
).fit()

print(model_home_no_conf.summary())
print(model_away_no_conf.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:             home_score   No. Observations:                16715
Model:                            GLM   Df Residuals:                    16710
Model Family:                 Poisson   Df Model:                            4
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -26848.
Date:                Thu, 02 Jul 2026   Deviance:                       22217.
Time:                        18:21:15   Pearson chi2:                 2.23e+04
No. Iterations:                     5   Pseudo R-squ. (CS):             0.3677
Covariance Type:            nonrobust                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                 1.0538      0.04

In [15]:

home_pred = model_home_no_conf.predict(X)
away_pred = model_away_no_conf.predict(X)

home_mae = mean_absolute_error(y_home, home_pred)
away_mae = mean_absolute_error(y_away, away_pred)

home_rmse = mean_squared_error(y_home, home_pred) ** 0.5
away_rmse = mean_squared_error(y_away, away_pred) ** 0.5

print("No-conf model validation")
print("-" * 30)
print(f"Home goals MAE: {home_mae:.3f}")
print(f"Away goals MAE: {away_mae:.3f}")
print(f"Home goals RMSE: {home_rmse:.3f}")
print(f"Away goals RMSE: {away_rmse:.3f}")

print("\nActual vs predicted means")
print(f"Actual home goals mean: {y_home.mean():.3f}")
print(f"Pred home goals mean:   {home_pred.mean():.3f}")
print(f"Actual away goals mean: {y_away.mean():.3f}")
print(f"Pred away goals mean:   {away_pred.mean():.3f}")


No-conf model validation
------------------------------
Home goals MAE: 1.091
Away goals MAE: 0.885
Home goals RMSE: 1.578
Away goals RMSE: 1.249

Actual vs predicted means
Actual home goals mean: 1.687
Pred home goals mean:   1.687
Actual away goals mean: 1.157
Pred away goals mean:   1.157


In [17]:
with open(EXPERIMENT_MODELS_DIR / "poisson_home.pkl", "wb") as f:
    pickle.dump(model_home_no_conf, f)

with open(EXPERIMENT_MODELS_DIR / "poisson_away.pkl", "wb") as f:
    pickle.dump(model_away_no_conf, f)

with open(EXPERIMENT_MODELS_DIR / "feature_columns.pkl", "wb") as f:
    pickle.dump(X.columns.tolist(), f)

print("Saved no-conf experiment models to:")
print(EXPERIMENT_MODELS_DIR)


Saved no-conf experiment models to:
/home/user1/project/fifa_prediction/models/experiments/no_conf


In [18]:
with open(EXPERIMENT_MODELS_DIR / "poisson_home.pkl", "rb") as f:
    loaded_home = pickle.load(f)

with open(EXPERIMENT_MODELS_DIR / "poisson_away.pkl", "rb") as f:
    loaded_away = pickle.load(f)

with open(EXPERIMENT_MODELS_DIR / "feature_columns.pkl", "rb") as f:
    loaded_feature_columns = pickle.load(f)

print(loaded_feature_columns)

test_pred_home = loaded_home.predict(X.head(3))
test_pred_away = loaded_away.predict(X.head(3))

print("Test home predictions:", test_pred_home.tolist())
print("Test away predictions:", test_pred_away.tolist())

['const', 'home_elo_pre', 'away_elo_pre', 'elo_diff', 'tournament_weight', 'neutral']
Test home predictions: [1.1711969453990516, 7.233057649097622, 1.2621470855797303]
Test away predictions: [0.9490063681821039, 0.24444530208127502, 1.1521272499406214]
